# 105: Cumulative Net-Negative CO2 Emissions

This notebook calculates the cumulative net-negative CO2 emissions for each scenario that achieves net-zero CO2.

**Definition:** Cumulative net-negative CO2 emissions are the integral of CO2 emissions from the year a scenario reaches net-zero CO2 until 2100.

**Inputs:**
- `101_harmonised_data.csv` - Harmonised emissions data
- `102_netzero_timings.csv` - Net-zero year metadata for each scenario

**Outputs:**
- `105_netzero_timings_cumnetnegco2.csv` - Scenario metadata with cumulative net-negative CO2 (Gt CO2)

In [1]:
import pyam
import pandas as pd

from pathlib import Path

/Users/ganti/Documents/code/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/jwt/api_jwt.py:147: InsecureKeyLengthWarning: The HMAC key is 20 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(


In [2]:
df = pyam.IamDataFrame(
    Path(
        "output/101_harmonised_data.csv"
    )
)

[INFO] 15:21:40 - pyam.core: Reading file output/101_harmonised_data.csv


## Load Data

Load net-zero timings as scenario metadata.

In [3]:
df.load_meta(
    Path(
        "output/102_netzero_timings.csv"
    )
)

# Calculate methane emissions reduction

In [4]:
df_ch4 = (
    df.filter(
        variable="Emissions|CH4",
        achieve_netzero_co2=True
    )
).timeseries()

In [5]:
ch4_decline = df_ch4.apply(
    lambda x: x[2023] - x[df.meta.loc[x.name[0:2], "year_netzero_co2"]],
    axis=1
).round(2)

## Filter and Aggregate CO2 Emissions

Filter for scenarios that achieve net-zero CO2.

In [6]:
df_netzero = (
    df
    .filter(achieve_netzero_co2=True)
)

Sum `Emissions|CO2|Fossil` and `Emissions|CO2|Biosphere` to get total `Emissions|CO2`.

In [7]:
df_co2 = df_netzero.add(
    a="Emissions|CO2|Fossil",
    b="Emissions|CO2|Biosphere",
    name='Emissions|CO2'
)

Convert units from Mt CO2/yr to Gt CO2/yr for cumulative calculation.

In [8]:
df_co2.convert_unit(
    current="Mt CO2/yr",
    to="Gt CO2/yr",
    inplace=True
)

In [9]:
df_co2.meta

year_netzero_co2  \
model             scenario                                    
AIM/CGE 2.0       SSP1-19                            2058.0   
                  SSP2-19                            2051.0   
                  SSP2-26                            2072.0   
                  SSP3-34                            2086.0   
                  SSP4-26                            2080.0   
...                                                     ...   
WITCH-GLOBIOM 4.4 CD-LINKS-INDC2030i_1600            2089.0   
                  CD-LINKS-NDC2030i_1000             2068.0   
                  CD-LINKS-NPi2020_1000              2079.0   
                  CD-LINKS-NPi2020_1600              2098.0   
                  CD-LINKS-NPi2020_400               2059.0   

                                           year_netzero_kyoto_ar6gwp100  \
model             scenario                                                
AIM/CGE 2.0       SSP1-19                                        2073.0   
                  SSP2-19                                        2060.0   
                  SSP2-26                                           NaN   
                  SSP3-34                                           NaN   
                  SSP4-26                                           NaN   
...                                                                 ...   
WITCH-GLOBIOM 4.4 CD-LINKS-INDC2030i_1600                           NaN   
                  CD-LINKS-NDC2030i_1000                         2080.0   
                  CD-LINKS-NPi2020_1000                          2093.0   
                  CD-LINKS-NPi2020_1600                             NaN   
                  CD-LINKS-NPi2020_400                           2071.0   

                                           achieve_netzero_co2  \
model             scenario                                       
AIM/CGE 2.0       SSP1-19                                 True   
                  SSP2-19                                 True   
                  SSP2-26                                 True   
                  SSP3-34                                 True   
                  SSP4-26                                 True   
...                                                        ...   
WITCH-GLOBIOM 4.4 CD-LINKS-INDC2030i_1600                 True   
                  CD-LINKS-NDC2030i_1000                  True   
                  CD-LINKS-NPi2020_1000                   True   
                  CD-LINKS-NPi2020_1600                   True   
                  CD-LINKS-NPi2020_400                    True   

                                           achieve_netzero_ghg  
model             scenario                                      
AIM/CGE 2.0       SSP1-19                                 True  
                  SSP2-19                                 True  
                  SSP2-26                                False  
                  SSP3-34                                False  
                  SSP4-26                                False  
...                                                        ...  
WITCH-GLOBIOM 4.4 CD-LINKS-INDC2030i_1600                False  
                  CD-LINKS-NDC2030i_1000                  True  
                  CD-LINKS-NPi2020_1000                   True  
                  CD-LINKS-NPi2020_1600                  False  
                  CD-LINKS-NPi2020_400                    True  

[743 rows x 4 columns]

## Calculate Cumulative Net-Negative Emissions

Use `pyam.timeseries.cumulative()` to integrate CO2 emissions from each scenario's net-zero year to 2100. This function performs linear interpolation between data points.

In [10]:
cumulative_netnegative_co2 = df_co2.timeseries().apply(
    lambda x: pyam.timeseries.cumulative(
        x,
        first_year=df_co2.meta.loc[x.name[0:2], "year_netzero_co2"],
        last_year=2100
    ).round(2),
    axis=1
)

Add cumulative net-negative CO2 as scenario metadata.

In [11]:
df_co2.set_meta(
    cumulative_netnegative_co2,
    name="cumulative_netnegative_co2"
)

In [12]:
# Add ch4_decline to metadata
df_co2.set_meta(ch4_decline, name="ch4_decline")

# Cut ch4_decline into three groups (terciles)
ch4_decline_groups = pd.qcut(
    ch4_decline, 
    q=3, 
    labels=["Low CH4 decline", "Medium CH4 decline", "High CH4 decline"]
)

# Add groups to metadata
df_co2.set_meta(ch4_decline_groups, name="ch4_decline_group")

print("CH4 decline group counts:")
print(ch4_decline_groups.value_counts().sort_index())

print("\nCH4 decline ranges per group:")
for group in ["Low CH4 decline", "Medium CH4 decline", "High CH4 decline"]:
    values = ch4_decline[ch4_decline_groups == group]
    print(f"  {group}: {values.min():.1f} to {values.max():.1f} Mt CH4/yr")

CH4 decline group counts:
Low CH4 decline       248
Medium CH4 decline    247
High CH4 decline      248
Name: count, dtype: int64

CH4 decline ranges per group:
  Low CH4 decline: -67.4 to 166.0 Mt CH4/yr
  Medium CH4 decline: 166.0 to 220.9 Mt CH4/yr
  High CH4 decline: 220.9 to 294.6 Mt CH4/yr


/Users/ganti/Documents/code/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/pyam/core.py:933: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  self.meta[name] = meta[name].combine_first(self.meta[name])


## Save Results

In [13]:
df_co2.meta.to_csv(
    Path(
        "output/105_netzero_timings_cumnetnegco2.csv"
    )
)